# Carvana: what happened to the cars we checked?

Read this notebook in order: **coverage -> website evidence -> changes -> attention**.
It follows two fixed groups of VINs. Some are historical controls; they help us
understand the website but cannot supply new sales for the inventory pilot.

**Run All reads saved files only.** A website Sold label is an observation, not a
verified delivery or a final sale after returns. Start with [Notebook 20](20_carvana_history_analysis.ipynb)
for inventory and asking-price changes. This notebook adds the vehicle-page evidence.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if (ROOT / 'vehicle/src').is_dir():
    ROOT = ROOT / 'vehicle'
elif ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))


## 1. Choose the evidence cutoff

The default cutoff is now. For a historical view, set `AS_OF_OVERRIDE` before
running this notebook. Use a timestamp with a timezone, such as
`2026-09-10T02:00:00Z`. All displayed timestamps are UTC; elapsed ages are hours.

| Time | Meaning |
| --- | --- |
| `selected_at` | When a VIN became part of a fixed cohort |
| `checked_at` | When someone actually inspected its page |
| `available_at` | When the saved observation became available to this project |

Only membership and observations known by the cutoff are used. The two cohort
files are read as saved; this notebook does not select new vehicles.
`SHOW_AUDIT_DETAILS` controls the longer source tables at the end.


In [ ]:
PILOT_AS_OF = globals().get('AS_OF_OVERRIDE', pd.Timestamp.now(tz='UTC').isoformat())
PILOT_RECHECK_HOURS = globals().get('PILOT_RECHECK_HOURS_OVERRIDE', 24)
PILOT_CONFIGS = [ROOT / 'config/carvana_sale_pilot.json',
                ROOT / 'config/carvana_sale_pilot_extension_20260909.json']
SHOW_AUDIT_DETAILS = globals().get('SHOW_AUDIT_DETAILS_OVERRIDE', False)
print('Evidence cutoff (UTC):', pd.Timestamp(PILOT_AS_OF).tz_convert('UTC').isoformat())
print('Frozen cohort sources:', *PILOT_CONFIGS, sep='\n')
print('Usable-native observation reminder interval, hours:', PILOT_RECHECK_HOURS)


In [ ]:
# Read-only replay; validate identities before any combined totals.
from vehicle_tracker.sale_pilot import known_disjoint_cohorts, load_pilot, summarize_pilot

sale_signal_study = pilot_observations = pilot_summary = pilot_queue = pd.DataFrame()
cohort_catalog = pd.DataFrame()
known_cohorts = []
alternative_view = any(name in globals() for name in [
    'CYCLE_REPORTS_OVERRIDE', 'TRACKING_CONFIG_OVERRIDE', 'DAILY_DATABASE_OVERRIDE',
    'DATABASE_OVERRIDE', 'CHECKS_OVERRIDE', 'REVIEWS_OVERRIDE', 'SALES_REVIEWS_OVERRIDE',
    'SELECTED_IDENTITY_OVERRIDE', 'CHECK_DRAFT_OVERRIDE'])
if alternative_view:
    print('Separate real pilot skipped for an alternative/synthetic analysis.')
else:
    missing_configs = [path for path in PILOT_CONFIGS if not path.is_file()]
    for path in missing_configs:
        print('Optional frozen cohort configuration is absent:', path, '| Its membership/coverage is unavailable.')
    frozen_cohorts = [json.loads(path.read_text(encoding='utf-8')) for path in PILOT_CONFIGS if path.is_file()]
    known_cohorts = known_disjoint_cohorts(frozen_cohorts, as_of=PILOT_AS_OF)
    known_ids = {cohort['cohort_id'] for cohort in known_cohorts}
    cohort_catalog = pd.DataFrame([dict(cohort_id=c['cohort_id'], selected_at=c['selected_at'],
        known_at_cutoff=c['cohort_id'] in known_ids) for c in frozen_cohorts])
    display(cohort_catalog)
    summaries, records = [], []
    for cohort in known_cohorts:
        observed = load_pilot(ROOT, cohort, as_of=PILOT_AS_OF)
        summary = summarize_pilot(observed, cohort, as_of=PILOT_AS_OF, recheck_hours=PILOT_RECHECK_HOURS)
        summary['cohort_id'] = cohort['cohort_id']
        summary['selected_at'] = cohort['selected_at']
        summary['selection_pending_group'] = summary.get('pending_group', pd.Series(index=summary.index, dtype='object')).fillna('not stratified')
        summaries.append(summary)
        if not observed.empty:
            records.append(observed.assign(cohort_id=cohort['cohort_id']))
    if summaries:
        pilot_summary = pd.concat(summaries, ignore_index=True)
        pilot_queue = pilot_summary[['cohort_id', 'retailer', 'vin', 'listing_id', 'url', 'role',
            'selection_pending_group', 'selection_reason', 'selected_at']].copy()
    if records:
        sale_signal_study = pd.concat(records, ignore_index=True)
        # Latest available interpretation per physical listing visit; source versions remain above.
        pilot_observations = sale_signal_study.sort_values(['checked_at', 'available_at']).drop_duplicates(
            ['cohort_id', 'retailer', 'vin', 'listing_id', 'checked_at'], keep='last')
    if not known_cohorts:
        print('No selected cohort was known at this cutoff. No outcome is assumed.')


## 2. Which selected vehicles were checked?

Start with the whole selected group, including VINs with no saved visit. The
original and extension cohorts remain separate. Their native pending=true/false
selection groups are diagnostics, not a representative sample of Carvana.

A visit can fail or leave purchase availability unclear. `usable_native_checks`
means the VIN matched and Carvana's native Available/Sold field was readable.
`usable_status_checks` also requires a resolved page interpretation. These overlap;
do not add them together. A pre-order page can be native Available but still
unresolved about purchase readiness.

`prospective_repeat_observed` counts initially non-Sold prospective VINs with a
later usable native check; historical controls do not count.
The complete calculations remain in `pilot_coverage`. Measured sales and outcome
rates stay missing: an unchecked VIN does not mean zero sales.


In [ ]:
pilot_coverage = combined_coverage = no_retained_check = pd.DataFrame()
if not pilot_summary.empty:
    coverage_rows = []
    groups = ['cohort_id', 'role', 'selection_pending_group']
    for key, part in pilot_summary.groupby(groups, dropna=False, sort=False):
        followed = int(part.prospective_repeat_observed.sum())
        coverage_rows.append(dict(zip(groups, key), selected_VINs=len(part),
            no_retained_check=int(part.checks.eq(0).sum()), one_check=int(part.checks.eq(1).sum()),
            repeated_checks=int(part.checks.gt(1).sum()), observed_VINs=int(part.checks.gt(0).sum()),
            physical_checks=int(part.checks.sum()), usable_status_checks=int(part.usable_status_checks.sum()),
            usable_native_checks=int(part.usable_native_checks.sum()), access_failures=int(part.access_failures.sum()),
            unresolved_checks=int(part.unresolved_checks.sum()),
            VINs_without_usable_status=int(part.usable_status_checks.eq(0).sum()),
            initially_already_Sold=int(part.first_encountered_sold.sum()),
            prospective_repeat_observed=followed,
            first_observed_Sold_transitions=int(part.newly_observed_sold.sum()) if followed else pd.NA,
            measured_sales=pd.NA, outcome_rate=pd.NA))
    pilot_coverage = pd.DataFrame(coverage_rows)
    print('Coverage by cohort, role and native pending selection group:')
    with pd.option_context('display.max_columns', None):
        display(pilot_coverage[groups + ['selected_VINs', 'no_retained_check', 'one_check',
            'repeated_checks', 'usable_native_checks', 'access_failures', 'unresolved_checks']])
    # Cohort identities were validated as disjoint and selected by the cutoff before concatenation.
    coverage_count_columns = ['selected_VINs', 'observed_VINs', 'no_retained_check', 'one_check',
        'repeated_checks', 'physical_checks', 'usable_status_checks', 'usable_native_checks',
        'access_failures', 'unresolved_checks', 'prospective_repeat_observed']
    combined_coverage = pilot_coverage[coverage_count_columns].sum().to_frame('known disjoint cohorts').T
    combined_coverage['measured_sales'] = pd.NA
    combined_coverage['outcome_rate'] = pd.NA
    print('Combined coverage only; read cohort-specific follow-up before discussing outcomes:')
    display(combined_coverage[['selected_VINs', 'observed_VINs', 'no_retained_check',
        'physical_checks', 'usable_native_checks', 'access_failures']])
    no_retained_check = pilot_queue.merge(pilot_summary[['cohort_id', 'retailer', 'vin', 'checks']],
        on=['cohort_id', 'retailer', 'vin'], validate='one_to_one').query('checks == 0')
    print('Unchecked VINs are kept in no_retained_check and the attention table below.')


## 3. What did each vehicle's page actually say?

`checked_vehicles` shows the **latest physical visit**, even if it failed. Native
`saleStatus` and `purchaseType` come from the target vehicle's public page data;
`latest_status` is our interpretation of those fields and its badge/button.
`why_unresolved` explains what the evidence still cannot establish.

| Native evidence | Interpretation |
| --- | --- |
| Sold with matching identity and consistent page evidence | `sold_label`: site-reported Sold |
| Available / NotPurchasable | `unavailable`: the cause is unknown |
| Available / Purchasable | `available` or `pending` only with a specific button/badge |
| Available / Reservable | Pre-order; native non-Sold, purchase readiness `unknown` |
| Failed access, conflicting identity or unrecognized fields | Status unresolved |

The parser checks URL, listing ID and VIN. It does not classify a recommendation
car or equipment text containing "as originally sold". Saved rows are selected
public-page projections, not original HTML or transaction records.


In [ ]:
latest_vehicle_status = checked_vehicles = pd.DataFrame()
if not pilot_summary.empty:
    latest_vehicle_status = pilot_summary.copy()
    latest_vehicle_status['why_unresolved'] = pd.NA
    latest_vehicle_status.loc[latest_vehicle_status.latest_status.eq('unavailable'), 'why_unresolved'] = (
        'The page is unavailable; it does not explain why.')
    unknown = latest_vehicle_status.latest_status.eq('unknown')
    latest_vehicle_status.loc[unknown, 'why_unresolved'] = 'No decisive purchase button or status badge.'
    latest_vehicle_status.loc[unknown & latest_vehicle_status.latest_purchaseType.eq('Reservable'), 'why_unresolved'] = (
        'Pre-order: native non-Sold, but purchase readiness is unclear.')
    failed = latest_vehicle_status.checks.gt(0) & latest_vehicle_status.latest_parse_outcome.ne('matched')
    latest_vehicle_status.loc[failed, 'why_unresolved'] = (
        'Check unresolved: ' + latest_vehicle_status.loc[failed, 'latest_parse_outcome'].fillna('unknown'))
    latest_vehicle_status.loc[latest_vehicle_status.checks.eq(0), 'why_unresolved'] = 'No retained page check.'
    checked_vehicles = latest_vehicle_status.loc[latest_vehicle_status.checks.gt(0)].copy()
    print('LATEST VISIT: website evidence, not completed sales. Full rows: checked_vehicles.')
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', 65):
        display(checked_vehicles[['vin', 'role', 'latest_listing_id', 'checked_at',
            'latest_saleStatus', 'latest_purchaseType', 'latest_status', 'why_unresolved']])
else:
    print('No cohort membership is available at this cutoff.')


## 4. What changed between checks?

`pilot_changes` compares the previous and latest physical visits. A failed or
conflicting check leaves change flags missing; it cannot turn an earlier value
into a new vehicle state. A changed interpretation can simply reflect a missing
button while the native fields stay unchanged. The before/after values remain
visible, so we can investigate rather than assume a cancellation or sale.

A **first observed Sold transition** additionally requires a prior matched native
non-Sold check and a first Sold check at/after selection. Historical controls and
vehicles already Sold on their first check cannot supply that transition.
`transition_interval_hours` is the time between the bounding observations, not a
delivery date. Later reappearance is also an observation, not proof of a return.


In [ ]:
pilot_changes = newly_observed_sold = initially_sold = later_reappearances = pd.DataFrame()
if not pilot_summary.empty:
    pilot_changes = pilot_summary.loc[pilot_summary.checks.gt(1)].copy()
    matched = pilot_changes.previous_parse_outcome.eq('matched') & pilot_changes.latest_parse_outcome.eq('matched')
    comparison_fields = [('saleStatus', 'previous_saleStatus', 'latest_saleStatus'),
                         ('purchaseType', 'previous_purchaseType', 'latest_purchaseType'),
                         ('interpretation', 'previous_status', 'latest_status')]
    change_columns = []
    for name, previous, latest in comparison_fields:
        column = name + '_changed'
        change_columns.append(column)
        pilot_changes[column] = pd.Series(pd.NA, index=pilot_changes.index, dtype='boolean')
        comparable = matched & pilot_changes[previous].notna() & pilot_changes[latest].notna()
        pilot_changes.loc[comparable, column] = pilot_changes.loc[comparable, previous].ne(pilot_changes.loc[comparable, latest])
    pilot_changes['change_observed'] = pilot_changes[change_columns].eq(True).any(axis=1)
    print('PREVIOUS / LATEST PHYSICAL VISITS: missing change flags mean not comparable.')
    with pd.option_context('display.max_rows', None):
        display(pilot_changes[['vin', 'previous_checked_at', 'checked_at',
            'previous_saleStatus', 'latest_saleStatus', 'previous_status', 'latest_status', *change_columns]])
    initially_sold = pilot_summary.loc[pilot_summary.first_encountered_sold,
        ['cohort_id', 'vin', 'role', 'first_sold_at', 'first_sold_listing_id']]
    newly_observed_sold = pilot_summary.loc[pilot_summary.newly_observed_sold,
        ['cohort_id', 'vin', 'role', 'selection_pending_group', 'last_non_sold_at', 'last_non_sold_listing_id',
         'first_sold_at', 'first_sold_listing_id', 'transition_interval_hours', 'reappeared_at', 'reappeared_listing_id']]
    print('Initially already-Sold vehicles (not new transitions):')
    display(initially_sold)
    print('Prospective VINs with a usable repeat check:', int(pilot_summary.prospective_repeat_observed.sum()))
    print('First qualifying Sold transitions; inspect follow-up coverage before interpreting an empty table:')
    display(newly_observed_sold)
    print('Read the observed/unobserved denominators above. This is not a count of actual sales.')
if not pilot_observations.empty:
    native_history = pilot_observations.loc[pilot_observations.parse_outcome.eq('matched')
        & pilot_observations.saleStatus.isin(['Available', 'Sold'])].sort_values(['checked_at', 'available_at']).copy()
    first_sold_time = native_history.checked_at.where(native_history.saleStatus.eq('Sold')).groupby(
        [native_history.cohort_id, native_history.retailer, native_history.vin]).transform('min')
    native_history['purchasable_or_reservable'] = (native_history.checked_at.gt(first_sold_time)
        & native_history.saleStatus.eq('Available')
        & native_history.purchaseType.isin(['Purchasable', 'Reservable']))
    previous_eligible = native_history.groupby(['cohort_id', 'retailer', 'vin']).purchasable_or_reservable.shift(fill_value=False)
    later_reappearances = native_history.loc[native_history.checked_at.gt(first_sold_time)
        & native_history.purchasable_or_reservable & ~previous_eligible,
        ['cohort_id', 'retailer', 'vin', 'listing_id', 'checked_at', 'available_at', 'saleStatus', 'purchaseType']]
    print('Later observed reappearances, preserving each listing ID:')
    display(later_reappearances)


## 5. What needs attention?

This is a **review list**, not an automatic collection command. It includes cars
with no check, failed/conflicting checks, changed website evidence, unresolved
page interpretations, or old native evidence. If several reasons apply, the
first reason in the visible `attention_order` list is shown.

`hours_since_usable_native` starts at the last matched native Available/Sold
observation. A recent failed visit does not refresh it. The 24-hour reminder is
an operating choice, not a sales rule. The full age diagnostics remain in
`pilot_freshness`; URLs in the queue are the original cohort URLs, with any later
checked listing ID shown separately.


In [ ]:
pilot_freshness = pilot_attention = pd.DataFrame()
if not pilot_summary.empty:
    pilot_freshness = pilot_summary[['cohort_id', 'retailer', 'vin', 'latest_listing_id',
        'last_attempt_at', 'hours_since_attempt', 'last_usable_native_at',
        'last_usable_native_listing_id', 'hours_since_usable_native',
        'no_usable_native_observation', 'overdue']].copy()
    attention = latest_vehicle_status.copy()
    attention['attention_reason'] = ''
    # Later assignments take precedence; the order below makes that priority explicit.
    attention.loc[attention.overdue, 'attention_reason'] = 'Native evidence is overdue'
    attention.loc[attention.latest_status.isin(['unknown', 'unavailable']), 'attention_reason'] = 'Review unresolved page evidence'
    if not pilot_changes.empty:
        changed = pilot_changes.loc[pilot_changes.change_observed, ['cohort_id', 'retailer', 'vin']].assign(website_changed=True)
        attention = attention.merge(changed, on=['cohort_id', 'retailer', 'vin'], how='left', validate='one_to_one')
        attention.loc[attention.website_changed.eq(True), 'attention_reason'] = 'Website evidence changed'
    attention.loc[attention.checks.gt(0) & attention.latest_parse_outcome.ne('matched'), 'attention_reason'] = 'Review failed or conflicting check'
    attention.loc[attention.checks.eq(0), 'attention_reason'] = 'No retained check yet'
    attention_order = ['No retained check yet', 'Review failed or conflicting check',
        'Website evidence changed', 'Review unresolved page evidence', 'Native evidence is overdue']
    attention['_order'] = attention.attention_reason.map({reason: i for i, reason in enumerate(attention_order)})
    pilot_attention = attention.loc[attention.attention_reason.ne('')].sort_values(
        ['_order', 'checked_at', 'vin'], na_position='first').drop(columns='_order')
    print('Review order:', ' -> '.join(attention_order))
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', 65):
        display(pilot_attention[['vin', 'role', 'listing_id', 'latest_listing_id', 'attention_reason',
            'hours_since_usable_native', 'url']])


The next manual step is to inspect this attention list and choose the next
checks from the fixed cohorts. A prior access challenge must be resolved before
resuming visits; a reminder is not permission to retry it. Keep failed and
unvisited cases visible.

Follow the [capture/import guide](../docs/sale_pilot.md) to capture a public page,
preview its VIN/listing identity and interpretation, and deliberately save the
observation. The importer preserves `checked_at` and records its actual import
availability. It does not write inventory, canonical checks or analyst reviews.
Run this notebook again to read the new evidence at a suitable cutoff.

No new visit is needed to use the tables above. A new website observation still
does not by itself establish delivery, completed sales, or forecast accuracy.


## 6. Inventory and source audit

The calculations below remain available for checking a surprising row.
Set `SHOW_AUDIT_DETAILS_OVERRIDE = True` before Run All to display the full tables.

`inventory_page_comparison` joins the latest page attempt to preceding inventory
by retailer/VIN. Both the inventory observation and its cycle availability must
precede the page check. Page and inventory listing IDs remain separate. A missing
date, partial cycle, or historical control outside the inventory scope cannot
establish absence or a sale; inspect `inventory_context` and the elapsed hours.

`pilot_queue` preserves fixed membership and selection reasons. `sale_signal_study`
contains all retained versions; `pilot_observations` uses the latest available
interpretation per physical visit. Failed checks keep their missing native fields.


In [ ]:
from vehicle_tracker.daily import tracking_settings, tracking_history
from vehicle_tracker.events import _aware, vin_events

inventory_cycles = inventory_rows = inventory_source_rows = pd.DataFrame()
inventory_page_comparison = inventory_comparison_diagnostics = pd.DataFrame()
inventory_config = ROOT / 'config/carvana_daily_tracking.json'
if not pilot_summary.empty:
    if inventory_config.is_file():
        inventory_settings = tracking_settings(inventory_config)
        inventory_cycles, inventory_rows = tracking_history(inventory_settings, as_of=PILOT_AS_OF)
    else:
        print('Optional inventory config absent; no inventory observation is assumed.')
    if not inventory_cycles.empty:
        inventory_cycles = inventory_cycles.loc[inventory_cycles.available_at.map(_aware).le(_aware(PILOT_AS_OF))].copy()
        inventory_source_rows = inventory_rows.merge(inventory_cycles[['cycle_id', 'available_at']],
            on='cycle_id', validate='many_to_one').rename(columns={'available_at': 'inventory_available_at'})
        inventory_source_rows['inventory_observed_at'] = inventory_source_rows.observed_at_utc.map(_aware)
        inventory_source_rows['inventory_available_at'] = inventory_source_rows.inventory_available_at.map(_aware)
    comparisons, diagnostics, validation = [], [], {}
    for page in pilot_summary.loc[pilot_summary.checks.gt(0)].itertuples():
        page_time = _aware(page.checked_at)
        item = dict(cohort_id=page.cohort_id, retailer=page.retailer, vin=page.vin, role=page.role,
            page_listing_id=page.latest_listing_id, page_checked_at=page_time,
            page_available_at=page.available_at, saleStatus=page.latest_saleStatus,
            purchaseType=page.latest_purchaseType, page_interpretation=page.latest_status,
            inventory_context='no_inventory_available_by_check', inventory_listing_id=None,
            inventory_observed_at=None, inventory_available_at=None, purchase_pending=None,
            asking_price_usd=None, elapsed_observation_hours=None, latest_inventory_cycle_date=None,
            latest_inventory_coverage_complete=None, inventory_source=None)
        if not inventory_cycles.empty:
            # Availability and physical clocks are separate restrictions, before any identity join.
            known_cycles = inventory_cycles.loc[inventory_cycles.available_at.map(_aware).le(page_time)]
            known_rows = inventory_source_rows.loc[inventory_source_rows.inventory_available_at.le(page_time)
                & inventory_source_rows.inventory_observed_at.le(page_time)]
            cycle_key = tuple(known_cycles.cycle_id)
            if cycle_key not in validation:
                try:
                    vin_events(known_cycles, known_rows)  # Reuse timeline's whole-population identity/window checks.
                    validation[cycle_key] = None
                except ValueError as error:
                    validation[cycle_key] = str(error)
            bindings = known_rows.loc[known_rows.retailer.eq(page.retailer) & known_rows.listing_id.eq(page.latest_listing_id)]
            problem = validation[cycle_key]
            if bindings.vin.ne(page.vin).any():
                problem = 'Page listing ID is bound to a different VIN in preceding inventory'
            if problem:
                item['inventory_context'] = 'invalid_identity_or_cycle_evidence'
                diagnostics.append(dict(cohort_id=page.cohort_id, vin=page.vin, problem=problem))
            elif not known_cycles.empty:
                latest_cycle = known_cycles.sort_values('cycle_date').iloc[-1]
                preceding = known_rows.loc[known_rows.retailer.eq(page.retailer) & known_rows.vin.eq(page.vin)]
                item.update(latest_inventory_cycle_date=latest_cycle.cycle_date,
                    latest_inventory_coverage_complete=latest_cycle.coverage_complete)
                if not preceding.empty:
                    previous = preceding.sort_values(['inventory_observed_at', 'inventory_available_at']).iloc[-1]
                    item.update(inventory_listing_id=previous.listing_id,
                        inventory_observed_at=previous.inventory_observed_at,
                        inventory_available_at=previous.inventory_available_at,
                        purchase_pending=previous.purchase_pending, asking_price_usd=previous.asking_price_usd,
                        elapsed_observation_hours=(page_time - previous.inventory_observed_at).total_seconds() / 3600,
                        inventory_source=previous.get('source_path', previous.get('source_url')))
                if preceding.empty and page.role == 'historical_control':
                    context = 'historical_control_not_observed_in_scope'
                elif latest_cycle.cycle_date != page_time.tz_convert(latest_cycle.timezone).date().isoformat():
                    context = 'collection_gap_unassessable'
                elif not latest_cycle.coverage_complete:
                    context = 'partial_cycle_unassessable'
                elif preceding.empty:
                    context = 'scope_membership_not_established'
                elif preceding.cycle_id.eq(latest_cycle.cycle_id).any():
                    context = 'observed_in_latest_complete_cycle'
                else:
                    context = 'not_observed_in_complete_cycle'
                item['inventory_context'] = context
        comparisons.append(item)
    inventory_page_comparison = pd.DataFrame(comparisons)
    inventory_comparison_diagnostics = pd.DataFrame(diagnostics)
    if SHOW_AUDIT_DETAILS:
        print('Preceding inventory and latest page attempt; both original clocks stay visible:')
        with pd.option_context('display.max_rows', None, 'display.max_columns', None):
            display(inventory_page_comparison)
    else:
        print('Preceding inventory audit is available in inventory_page_comparison.')
    if not inventory_comparison_diagnostics.empty:
        display(inventory_comparison_diagnostics)


In [ ]:
if SHOW_AUDIT_DETAILS:
    for cohort in known_cohorts:
        print('FROZEN MEMBERSHIP:', cohort['cohort_id'])
        with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
            display(pilot_queue.loc[pilot_queue.cohort_id.eq(cohort['cohort_id'])])
        if 'selection' in cohort:
            print('Frozen sampling metadata, not a new selection:')
            selection = cohort['selection']
            selection_age_hours = (pd.Timestamp(cohort['selected_at']) - pd.Timestamp(selection['source']['window_end'])).total_seconds() / 3600
            display(pd.DataFrame([dict(seed=selection['seed'], method=selection['method'],
                pandas_version=selection['pandas_version'], source_age_at_selection_hours=selection_age_hours)]))
            display(pd.DataFrame(selection['counts']))
            display(pd.DataFrame([cohort['selection']['source']]))
    if not sale_signal_study.empty:
        print('All retained versions at the cutoff; physical visits are deduplicated for coverage:')
        with pd.option_context('display.max_rows', None, 'display.max_columns', None):
            display(sale_signal_study[['cohort_id', 'retailer', 'vin', 'listing_id', 'observed_listing_id',
                'observed_vin', 'checked_at', 'available_at', 'saleStatus', 'purchaseType', 'inventoryType',
                'hero_badge', 'purchase_button', 'observed_status', 'access_outcome', 'parse_outcome', 'source']])
    else:
        print('No retained observations for the known cohorts at this cutoff.')
else:
    print('Membership and full source rows remain available in pilot_queue and sale_signal_study.')
